# Étape 2 — Optimisation (balayage de configurations, métrique **officielle**)

Baseline obtenue (`01_baseline_bpe_10k.ipynb`) : **score officiel = 2.059977**, guardrails EN/FR **PASS**.

Ce notebook cherche à **faire baisser ce score** en testant plusieurs configurations, en utilisant
**exactement la métrique officielle du challenge** (lue dans le dépôt officiel
[`airf-multilingual-tokenizer-challenge`](https://github.com/aims-ai-research-foundations/airf-multilingual-tokenizer-challenge), `competition/metrics.py` + `competition/constants.py`).

## Règles officielles (vérifiées dans le code du challenge)

| Élément | Valeur officielle |
|---|---|
| Mot | `len(text.split())` (espaces blancs) |
| Fertility | `tokens / words` |
| Pénalité UNK | `100 × (unk_tokens / words)` |
| Score par langue | `fertility + 100 × unk_rate` |
| Score final | moyenne de **ha, sw, yo, am** |
| Guardrail EN/FR | `budget = 1.15 × moyenne(fertility brute ha,sw,yo,am)` ; échec si `fertility(en)` **ou** `fertility(fr)` > budget |
| Vocabulaire max | 10 000 (`get_vocab_size(with_added_tokens=True)`) |
| Taille max du fichier | 20 MiB |
| Version `tokenizers` | **`0.22.1` exactement** (sinon `compatible_version` échoue) |
| Données | **uniquement** le `train` fourni par le challenge |

> ⚠️ **Point stratégique majeur** : le budget du guardrail est **relatif** — il vaut 1,15 × la moyenne
> des langues africaines **de votre propre tokenizer**. Donc si l'on améliore beaucoup ha/sw/yo/am
> sans améliorer en/fr, le budget baisse et le guardrail peut **casser** (soumission rejetée).
> Chaque configuration est donc évaluée **avec son verdict de guardrail**.

## Axes testés (7–8 configurations)

1. **`b1-baseline`** — référence : `BPE + NFC + Whitespace`, vocab 10k, `min_frequency=2` (doit redonner ≈ 2.0600).
2. **`c2-wssplit-mf2`** — pré-tokeniseur `WhitespaceSplit` : la ponctuation **reste collée** au mot
   (`shared.` = 1 pré-token au lieu de 2). Le baseline isole `,` et `.` qui représentaient à eux seuls
   ~43 000 tokens sur la validation.
3. **`c3-wssplit-mf5`** — idem + `min_frequency=5` (moins de fusions rares gaspillées).
4. **`c4-wssplit-mf10-alpha`** — idem + `min_frequency=10` + **alphabet initial = tous les caractères du train**
   (supprime les `[UNK]` pour tout caractère vu à l'entraînement).
5. **`c5-bytelevel`** — `ByteLevel(use_regex=True)` + `BPE` : round-trip **sans perte** et **zéro `[UNK]`**,
   mais coût en octets pour l'éthiopien (3 octets/caractère) → à mesurer.
6. **`c6-unigram-alpha`** — modèle **Unigram** (souvent meilleur à vocabulaire fixe) + alphabet complet.
7. **`c7-...-amboost`** — comme c4 + **sur-échantillonnage de l'amharique (×2)** : l'amharique est la pire
   langue (fertility 2,49 / 130 `[UNK]`) ; on lui alloue plus de fusions.
8. **`c8-scoredboost`** — sur-échantillonnage des **4 langues notées** (×2) : teste la limite du guardrail
   (baisse le score mais peut faire échouer EN/FR — c'est justement ce que l'on veut mesurer).

À la fin : tableau classé, verdict guardrail, **sauvegarde du meilleur**, rapport
`reports/optimization_sweep.{json,md}`, dossier de soumission `submissions/<slug>/` et
exécution du **checker officiel** (`starter/utils.py`).

## 1. Installation (versions officielles)

`tokenizers==0.22.1` est **imposé** par le challenge : le checker officiel vérifie l'égalité exacte
de version (`SUPPORTED_TOKENIZERS_VERSION = "0.22.1"`). On épingle donc cette version.

In [ ]:
!pip install -q "tokenizers==0.22.1" datasets pandas numpy
import tokenizers
print("tokenizers:", tokenizers.__version__, "(attendu 0.22.1)")
assert tokenizers.__version__ == "0.22.1", "Installer tokenizers==0.22.1 (exigence officielle)"

## 2. Constantes, métrique officielle et guardrail

Le code ci-dessous reproduit **exactement** la métrique officielle
(`competition/metrics.py`) et les constantes (`competition/constants.py`).

In [ ]:
# =============================================================================
# 2. Constantes + métrique OFFICIELLES (compétition)
# =============================================================================
import json, os, re, shutil, time, unicodedata
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd
from tokenizers import Tokenizer

# ---- Constantes officielles (competition/constants.py) ---------------------
LANGUAGES = ("en", "fr", "ha", "sw", "yo", "am")
LANGUAGE_NAMES = {"en": "English", "fr": "French", "ha": "Hausa",
                  "sw": "Swahili", "yo": "Yoruba", "am": "Amharic"}
SCORED_LANGUAGES = ("ha", "sw", "yo", "am")
CONTEXT_LANGUAGES = ("en", "fr")
CONTEXT_FERTILITY_RATIO = 1.15
UNKNOWN_PENALTY = 100.0
MAX_VOCAB_SIZE = 10_000
MAX_TOKENIZER_BYTES = 20 * 1024 * 1024
REQUIRED_TOKENIZERS_VERSION = "0.22.1"
SMOKE_TEXTS = {
    "en": "Knowledge grows when it is shared.",
    "fr": "Le savoir grandit lorsqu’il est partagé.",
    "ha": "Ilimi yana ƙaruwa idan an raba shi.",
    "sw": "Maarifa hukua yanaposhirikishwa.",
    "yo": "Ìmọ̀ ń pọ̀ sí i nígbà tí a bá pín in.",
    "am": "እውቀት ሲካፈል ያድጋል።",
}

# ---- Dataset officiel ------------------------------------------------------
DATASET_NAME = "Similoluwa/african-multilingual-tokenizer-challenge"
DATASET_REVISION = "v1.0.0"

# ---- Paramètres du balayage (modifiables) ----------------------------------
VOCAB_SIZE = 10_000            # imposé par le challenge
MAX_TRAIN_DOCS = None          # None = tout le train (240 000) ; ex. 60_000 pour un pré-balayage rapide
BASELINE_REFERENCE_SCORE = 2.059977   # score officiel obtenu par 01_baseline_bpe_10k

OUTPUT_ROOT = Path.cwd()
REPORT_DIR = OUTPUT_ROOT / "reports"
MODEL_DIR = OUTPUT_ROOT / "models"
SUBMISSIONS_DIR = OUTPUT_ROOT / "submissions"
for d in (REPORT_DIR, MODEL_DIR, SUBMISSIONS_DIR):
    d.mkdir(parents=True, exist_ok=True)


# ---- MÉTRIQUE OFFICIELLE ---------------------------------------------------
def count_words(text: str) -> int:
    """Mots = séparés par des espaces blancs (comme l'évaluateur officiel)."""
    return len(text.split())


def unknown_token_id(tokenizer: Tokenizer):
    """Id émis pour un texte non représentable (comme l'évaluateur officiel)."""
    model = json.loads(tokenizer.to_str()).get("model", {})
    name = model.get("unk_token")
    if isinstance(name, str):
        return tokenizer.token_to_id(name)
    unk_id = model.get("unk_id")
    return int(unk_id) if unk_id is not None else None


def measure(rows, tokenizer, batch_size=2048):
    """rows = liste de (language, text) -> fertility, unk_rate, tokens, words, lossy, unk_total."""
    tokens, words, unknowns = defaultdict(int), defaultdict(int), defaultdict(int)
    unknown_id = unknown_token_id(tokenizer)
    lossy = 0
    for start in range(0, len(rows), batch_size):
        batch = rows[start:start + batch_size]
        encodings = tokenizer.encode_batch([t for _, t in batch], add_special_tokens=False)
        for (lang, text), enc in zip(batch, encodings, strict=True):
            tokens[lang] += len(enc.ids)
            words[lang] += count_words(text)
            if unknown_id is not None:
                unknowns[lang] += sum(1 for v in enc.ids if v == unknown_id)
            if tokenizer.decode(enc.ids, skip_special_tokens=False) != text:
                lossy += 1
    fertility = {l: tokens[l] / words[l] for l in LANGUAGES if words[l]}
    unk_rate = {l: unknowns[l] / words[l] for l in LANGUAGES if words[l]}
    return fertility, unk_rate, dict(tokens), dict(words), lossy, sum(unknowns.values())


def penalised_scores(fertility, unk_rate):
    return {l: v + UNKNOWN_PENALTY * unk_rate.get(l, 0.0) for l, v in fertility.items()}


def competition_score(fertility, unk_rate=None):
    """Score officiel = moyenne des langues notées (ha, sw, yo, am)."""
    missing = [l for l in SCORED_LANGUAGES if l not in fertility]
    if missing:
        raise ValueError(f"langues notées manquantes : {missing}")
    scores = penalised_scores(fertility, unk_rate or {})
    return sum(scores[l] for l in SCORED_LANGUAGES) / len(SCORED_LANGUAGES)


def guardrail(fertility):
    """budget = 1.15 x moyenne(fertility brute des langues notées)."""
    raw = sum(fertility[l] for l in SCORED_LANGUAGES) / len(SCORED_LANGUAGES)
    budget = raw * CONTEXT_FERTILITY_RATIO
    breaches = [l for l in CONTEXT_LANGUAGES if fertility.get(l, 0.0) > budget]
    return raw, budget, breaches


def validate_tokenizer_file(path):
    """Contrôles officiels de validation (competition/validation.py)."""
    path = Path(path)
    checks, errors = {}, []
    checks["file_size"] = path.stat().st_size <= MAX_TOKENIZER_BYTES
    if not checks["file_size"]:
        errors.append("fichier > 20 MiB")
    tok = Tokenizer.from_file(str(path))
    checks["loads"] = True
    vocab_size = tok.get_vocab_size(with_added_tokens=True)
    checks["vocabulary"] = vocab_size <= MAX_VOCAB_SIZE
    if not checks["vocabulary"]:
        errors.append(f"vocabulaire {vocab_size:,} > {MAX_VOCAB_SIZE:,}")
    encodings = tok.encode_batch(list(SMOKE_TEXTS.values()), add_special_tokens=False)
    checks["encodes_all_languages"] = all(e.ids for e in encodings)
    if not checks["encodes_all_languages"]:
        errors.append("une langue ne produit aucun token")
    decoded = [tok.decode(e.ids, skip_special_tokens=False) for e in encodings]
    checks["decodes"] = all(t.strip() for t in decoded)
    if not checks["decodes"]:
        errors.append("un décodage est vide")
    import tokenizers as _tk
    checks["compatible_version"] = _tk.__version__ == REQUIRED_TOKENIZERS_VERSION
    if not checks["compatible_version"]:
        errors.append(f"tokenizers {_tk.__version__} != {REQUIRED_TOKENIZERS_VERSION}")
    lossy_languages = [l for l, orig, rest in zip(SMOKE_TEXTS, SMOKE_TEXTS.values(), decoded)
                       if orig != rest]
    return {"valid": all(checks.values()) and not errors, "checks": checks, "errors": errors,
            "vocab_size": vocab_size, "file_size_bytes": path.stat().st_size,
            "lossy_languages": lossy_languages}


print("Métrique officielle chargée. Vocab max:", MAX_VOCAB_SIZE, "| guardrail ratio:", CONTEXT_FERTILITY_RATIO)
print("Tokenizers:", tokenizers.__version__ if (tokenizers := __import__("tokenizers")) else None)

## 3. Données : chargement + préparation

Entraînement sur `train` uniquement, évaluation sur `validation` uniquement (comme le baseline).

In [ ]:
# =============================================================================
# 3. Chargement du dataset officiel + préparation des textes
# =============================================================================
from datasets import load_dataset

dataset = load_dataset(DATASET_NAME, revision=DATASET_REVISION)
print(dataset)

train_by_lang = {}
for lang in LANGUAGES:
    train_by_lang[lang] = [t for t, l in zip(dataset["train"]["text"], dataset["train"]["language"]) if l == lang]
if MAX_TRAIN_DOCS:
    train_by_lang = {l: texts[:MAX_TRAIN_DOCS] for l, texts in train_by_lang.items()}

val_rows = []
for lang in LANGUAGES:
    texts = [t for t, l in zip(dataset["validation"]["text"], dataset["validation"]["language"]) if l == lang]
    val_rows.extend((lang, t) for t in texts)

print("\nEntraînement :", {l: f"{len(v):,}" for l, v in train_by_lang.items()},
      "| total:", f"{sum(len(v) for v in train_by_lang.values()):,}")
print("Validation   :", {l: f"{sum(1 for x, _ in val_rows if x == l):,}" for l in LANGUAGES},
      "| total:", f"{len(val_rows):,}")
assert len(dataset["train"]) == 240_000 or MAX_TRAIN_DOCS, "train inattendu"
assert len(val_rows) == 24_000, "validation inattendue"

## 4. Configurations candidates

Chaque configuration est décrite par un dict : modèle, normaliseur, pré-tokeniseur, `min_frequency`,
alphabet initial (complet / bytes), sur-échantillonnage éventuel et, pour `c9-bytefallback`,
le **byte fallback** (`byte_fallback=True`).

**EXP-004 — `c9-bytefallback`** : reprend `c8-scoredboost` et ajoute le byte fallback. Les 256
tokens `<0xXX>` couvrent **tout** caractère possible (UTF-8), donc les 255 `[UNK]` de la
validation disparaissent — sans utiliser la moindre donnée supplémentaire, puisque c'est un
changement d'algorithme, pas de vocabulaire appris. Coût : 256 places de vocabulaire en moins
pour les merges appris, et 2 à 4 tokens par caractère rare au lieu d'un seul `[UNK]`.

In [ ]:
# =============================================================================
# 4. Définition des configurations candidates
# =============================================================================
from tokenizers.models import BPE, Unigram
from tokenizers.normalizers import NFC
from tokenizers.pre_tokenizers import ByteLevel as ByteLevelPre, Whitespace, WhitespaceSplit
from tokenizers.decoders import ByteFallback
from tokenizers.decoders import ByteLevel as ByteLevelDec
from tokenizers.trainers import BpeTrainer, UnigramTrainer

# --- EXP-004 : byte fallback ---------------------------------------------------
# Recette mesurée : les 256 tokens <0xXX> doivent être DANS le vocabulaire du modèle (sinon le
# byte_fallback du modèle ne trouve rien et émet [UNK]) et comptent DANS vocab_size.
# Passer par add_tokens() après entraînement NE fonctionne PAS (mesuré : 41/41 mots -> [UNK]).
BYTE_TOKENS = [f"<0x{i:02X}>" for i in range(256)]


def is_byte_token(token):
    """Vrai pour les tokens byte du type <0xEF> (tokens ordinaires, comme dans Llama-2)."""
    return len(token) == 6 and token.startswith("<0x") and token.endswith(">")


CONFIGS = [
    dict(name="b1-baseline", model="bpe", pre="whitespace", min_freq=2,
         alphabet=False, boost={},
         note="référence : BPE+NFC+Whitespace, mf=2 (doit redonner ~2.0600)"),
    dict(name="c2-wssplit-mf2", model="bpe", pre="whitespace_split", min_freq=2,
         alphabet=False, boost={},
         note="ponctuation collée au mot (WhitespaceSplit)"),
    dict(name="c3-wssplit-mf5", model="bpe", pre="whitespace_split", min_freq=5,
         alphabet=False, boost={},
         note="WhitespaceSplit + min_frequency=5"),
    dict(name="c4-wssplit-mf10-alpha", model="bpe", pre="whitespace_split", min_freq=10,
         alphabet=True, boost={},
         note="WhitespaceSplit + mf=10 + alphabet complet (supprime les UNK connus)"),
    dict(name="c5-bytelevel", model="bpe", pre="byte_level", min_freq=2,
         alphabet="bytes", boost={},
         note="ByteLevel(use_regex) : round-trip sans perte, zero UNK"),
    dict(name="c6-unigram-alpha", model="unigram", pre="whitespace_split", min_freq=2,
         alphabet=True, boost={},
         note="modele Unigram + alphabet complet"),
    dict(name="c7-wssplit-mf10-alpha-amboost", model="bpe", pre="whitespace_split", min_freq=10,
         alphabet=True, boost={"am": 2},
         note="comme c4 + amharique sur-echantillonne x2"),
    dict(name="c8-scoredboost", model="bpe", pre="whitespace_split", min_freq=5,
         alphabet=True, boost={"ha": 2, "sw": 2, "yo": 2, "am": 2},
         note="sur-echantillonnage des 4 langues notees (teste la limite du guardrail)"),
    dict(name="c9-bytefallback", model="bpe", pre="whitespace_split", min_freq=5,
         alphabet=True, boost={"ha": 2, "sw": 2, "yo": 2, "am": 2}, byte_fallback=True,
         note="EXP-004 : c8 + byte_fallback (256 tokens <0xXX>) -> supprime les [UNK]"),
]

# Alfabet : caractères du train (>=1 occurrence) pour 'alphabet=True'
if any(c["alphabet"] is True for c in CONFIGS):
    train_chars = set()
    for texts in train_by_lang.values():
        for t in texts:
            train_chars.update(ch for ch in t if not ch.isspace())
    train_alphabet = sorted(train_chars)
    print(f"Alphabet du train : {len(train_alphabet):,} caractères distincts (hors espaces)")
else:
    train_alphabet = []

# Tirage : permet de lancer un sous-ensemble  ->  RUN_CONFIGS = {"c4-wssplit-mf10-alpha"}
RUN_CONFIGS = None            # None = toutes les configurations
selected = [c for c in CONFIGS if RUN_CONFIGS is None or c["name"] in RUN_CONFIGS]
print(f"\n{len(selected)} configuration(s) à entraîner :")
for c in selected:
    print(f"  - {c['name']:32s} {c['note']}")

## 5. Entraînement + évaluation de chaque configuration

Pour chaque configuration : entraînement sur `train`, puis évaluation **sur `validation`** avec la
métrique officielle (score, fertility par langue, UNK, **verdict guardrail**).

⏱️ Durée indicative dans Colab : quelques minutes par configuration (8 configurations ≈ 20–40 min).
Utiliser `MAX_TRAIN_DOCS = 60_000` dans la cellule 2 pour un pré-balayage rapide, puis relancer les
2–3 meilleures en données complètes.

In [ ]:
# =============================================================================
# 5. Balayage : entraînement + évaluation officielle
# =============================================================================
def corpus_iterator(train_by_lang, boost=None, log_every=50_000):
    """Itère les textes multilingues en round-robin (équilibré) avec sur-échantillonnage."""
    boost = boost or {}
    iters = {l: iter(texts) for l, texts in train_by_lang.items()}
    active = list(train_by_lang)
    i = 0
    while active:
        for lang in list(active):
            try:
                text = next(iters[lang])
            except StopIteration:
                active.remove(lang)
                continue
            for _ in range(boost.get(lang, 1)):
                yield text
                i += 1
                if log_every and i % log_every == 0:
                    print(f"    ... {i:,} textes fournis")


def strip_byte_added_tokens(tokenizer):
    """Rend les tokens byte ordinaires (forme des tokenizers Llama-2).

    Le trainer les ajoute via special_tokens (seul moyen mesuré de les faire entrer dans le
    vocabulaire du modèle) : on retire leur entrée `added_tokens`, ils restent des tokens du
    modèle BPE — byte fallback intact, et decode(skip_special_tokens=True) ne les jette plus.
    """
    payload = json.loads(tokenizer.to_str())
    payload["added_tokens"] = [t for t in payload["added_tokens"]
                               if not is_byte_token(t["content"])]
    return Tokenizer.from_str(json.dumps(payload))


def build_tokenizer(cfg):
    """Construit et entraîne un tokenizer selon la configuration (utilise train seulement)."""
    byte_fallback = bool(cfg.get("byte_fallback", False))
    if cfg["model"] == "bpe":
        tokenizer = Tokenizer(BPE(unk_token="[UNK]", byte_fallback=byte_fallback))
    else:
        tokenizer = Tokenizer(Unigram())
    tokenizer.normalizer = NFC()

    if cfg["pre"] == "whitespace":
        tokenizer.pre_tokenizer = Whitespace()
    elif cfg["pre"] == "whitespace_split":
        tokenizer.pre_tokenizer = WhitespaceSplit()
    elif cfg["pre"] == "byte_level":
        tokenizer.pre_tokenizer = ByteLevelPre(add_prefix_space=False, use_regex=True)
        tokenizer.decoder = ByteLevelDec()

    if cfg["alphabet"] == "bytes":
        alphabet = ByteLevelPre.alphabet()
    elif cfg["alphabet"] is True:
        alphabet = train_alphabet
    else:
        alphabet = None

    if cfg["model"] == "bpe":
        specials = ["[UNK]"] + (BYTE_TOKENS if byte_fallback else [])
        kwargs = dict(vocab_size=VOCAB_SIZE, min_frequency=cfg["min_freq"], special_tokens=specials)
        if alphabet:
            kwargs["initial_alphabet"] = alphabet
        trainer = BpeTrainer(**kwargs)
    else:
        kwargs = dict(vocab_size=VOCAB_SIZE, special_tokens=["[UNK]"], unk_token="[UNK]")
        if alphabet:
            kwargs["initial_alphabet"] = alphabet
        trainer = UnigramTrainer(**kwargs)

    tokenizer.train_from_iterator(
        corpus_iterator(train_by_lang, cfg.get("boost")), trainer=trainer)

    if byte_fallback:
        tokenizer.decoder = ByteFallback()
        tokenizer = strip_byte_added_tokens(tokenizer)
        print(f"    byte_fallback : {len(BYTE_TOKENS)} tokens <0xXX> dans le vocabulaire "
              f"(total {tokenizer.get_vocab_size(with_added_tokens=True):,}), "
              f"added_tokens restants = {len(json.loads(tokenizer.to_str())['added_tokens'])}")
    return tokenizer


results = []
for n, cfg in enumerate(selected, start=1):
    print(f"\n{'='*84}\n[{n}/{len(selected)}] {cfg['name']} — {cfg['note']}\n{'='*84}")
    t0 = time.time()
    tok = build_tokenizer(cfg)
    train_seconds = time.time() - t0

    fertility, unk_rate, tokens, words, lossy, unk_total = measure(val_rows, tok)
    score = competition_score(fertility, unk_rate)
    raw, budget, breaches = guardrail(fertility)
    penalised = penalised_scores(fertility, unk_rate)
    vocab_size = tok.get_vocab_size(with_added_tokens=True)

    results.append(dict(
        name=cfg["name"], note=cfg["note"], config=cfg,
        vocab_size=vocab_size, train_seconds=train_seconds,
        score=score, fertility=fertility, unk_rate=unk_rate, penalised=penalised,
        tokens=tokens, words=words, unk_total=unk_total, lossy_rows=lossy,
        raw_scored=raw, guardrail_budget=budget, guardrail_breaches=breaches,
        guardrail_pass=not breaches, tokenizer=tok,
    ))
    flag = "GUARDRAIL OK" if not breaches else f"GUARDRAIL FAIL {breaches}"
    print(f"  score={score:.4f} | vocab={vocab_size} | UNK={unk_total} | lossy={lossy:,} | "
          f"{flag} | {train_seconds/60:.1f} min")

print("\nBalayage terminé.")

## 6. Résultats classés

Tableau trié par score officiel (plus bas = meilleur), avec le verdict guardrail de chaque configuration.

In [ ]:
# =============================================================================
# 6. Tableau comparatif + classement
# =============================================================================
rows = []
for r in sorted(results, key=lambda x: x["score"]):
    rows.append({
        "config": r["name"],
        "Score ↓": round(r["score"], 4),
        "Guardrail": "PASS" if r["guardrail_pass"] else "FAIL",
        "Hausa": round(r["penalised"]["ha"], 4),
        "Swahili": round(r["penalised"]["sw"], 4),
        "Yoruba": round(r["penalised"]["yo"], 4),
        "Amharic": round(r["penalised"]["am"], 4),
        "UNK": r["unk_total"],
        "lossy": f"{r['lossy_rows']:,}",
        "vocab": r["vocab_size"],
        "en": round(r["fertility"]["en"], 4),
        "fr": round(r["fertility"]["fr"], 4),
        "budget": round(r["guardrail_budget"], 4),
    })
sweep_df = pd.DataFrame(rows)
print("Score officiel = moyenne des scores (ha, sw, yo, am). Lower is better.")
print("Guardrail = fertility(en) et fertility(fr) <= 1.15 x moyenne brute des 4 langues notées.\n")
print(sweep_df.to_string(index=False))

valid_results = [r for r in results if r["guardrail_pass"]]
best = min(valid_results, key=lambda r: r["score"]) if valid_results else None

print("\n--- Comparaison à la baseline (2.059977) ---")
baseline_result = next((r for r in results if r["name"] == "b1-baseline"), None)
if baseline_result:
    delta_check = baseline_result["score"] - BASELINE_REFERENCE_SCORE
    print(f"b1-baseline reproduit {baseline_result['score']:.4f} "
          f"(référence {BASELINE_REFERENCE_SCORE:.4f}, écart {delta_check:+.4f})")
if best:
    gain = BASELINE_REFERENCE_SCORE - best["score"]
    print(f"\nMEILLEURE CONFIGURATION ÉLIGIBLE : {best['name']}")
    print(f"  score {best['score']:.4f}  (gain vs baseline : {gain:+.4f} soit {100*gain/BASELINE_REFERENCE_SCORE:.1f} %)")
    print(f"  guardrail : budget {best['guardrail_budget']:.4f} | en {best['fertility']['en']:.4f} | fr {best['fertility']['fr']:.4f}")
    for l in LANGUAGES:
        print(f"    {l}: fertility {best['fertility'][l]:.4f} | unk_rate {best['unk_rate'][l]:.6f} | score {best['penalised'][l]:.4f}")
else:
    print("Aucune configuration ne passe le guardrail : revoir les configurations.")

## 7. Analyse automatique : quel levier a fonctionné ?

Comparaison des axes testés pour comprendre **pourquoi** une configuration gagne.

In [ ]:
# =============================================================================
# 7. Analyse des leviers
# =============================================================================
base = next((r for r in results if r["name"] == "b1-baseline"), None)
print("Effet des leviers (score relatif à la baseline) :\n")
for r in sorted(results, key=lambda x: x["score"]):
    rel = "" if base is None else f"{r['score'] - base['score']:+.4f}"
    print(f"  {r['name']:34s} {r['score']:.4f}  ({rel})")

print("\nLecture des UNK :")
for r in results:
    print(f"  {r['name']:34s} UNK total={r['unk_total']:>6} | " +
          " ".join(f"{l}:{r['unk_rate'][l]*100:.3f}%" for l in LANGUAGES))

print("\nLecture de la fertility par langue (brute) :")
print(f"  {'config':34s} " + " ".join(f"{l:>8s}" for l in LANGUAGES))
for r in results:
    print(f"  {r['name']:34s} " + " ".join(f"{r['fertility'][l]:>8.4f}" for l in LANGUAGES))

## 8. Sauvegarde du meilleur modèle + rapports

Le tokenizer gagnant est écrit dans `models/optimized_<config>/tokenizer.json` et les rapports dans
`reports/optimization_sweep.{json,md}`.

In [ ]:
# =============================================================================
# 8. Sauvegarde du meilleur + rapports JSON/Markdown
# =============================================================================
best_model_dir = MODEL_DIR / f"optimized_{best['name']}"
best_model_dir.mkdir(parents=True, exist_ok=True)
best_tokenizer_path = best_model_dir / "tokenizer.json"
best["tokenizer"].save(str(best_tokenizer_path))
print("Meilleur tokenizer sauvegardé :", best_tokenizer_path,
      f"({best_tokenizer_path.stat().st_size:,} octets)")

# Copie "candidate" à la racine pour le checker officiel
candidate_path = OUTPUT_ROOT / "tokenizer.json"
shutil.copy2(best_tokenizer_path, candidate_path)
print("Candidat pour le checker :", candidate_path)

report = {
    "experiment": "optimization_sweep",
    "dataset": {"name": DATASET_NAME, "revision": DATASET_REVISION,
                "train_docs_used": {l: len(v) for l, v in train_by_lang.items()},
                "validation_rows": len(val_rows)},
    "official_rules": {
        "scored_languages": list(SCORED_LANGUAGES),
        "context_languages": list(CONTEXT_LANGUAGES),
        "guardrail_ratio": CONTEXT_FERTILITY_RATIO,
        "unknown_penalty": UNKNOWN_PENALTY,
        "max_vocab_size": MAX_VOCAB_SIZE,
        "required_tokenizers_version": REQUIRED_TOKENIZERS_VERSION,
        "word_definition": "len(text.split())",
    },
    "settings": {"vocab_size": VOCAB_SIZE, "max_train_docs": MAX_TRAIN_DOCS},
    "baseline_reference_score": BASELINE_REFERENCE_SCORE,
    "results": [
        {k: v for k, v in r.items() if k not in ("tokenizer",)}
        for r in sorted(results, key=lambda x: x["score"])
    ],
    "best": {"name": best["name"], "score": best["score"], "config": best["config"],
             "guardrail_pass": best["guardrail_pass"], "model_path": str(best_tokenizer_path)},
    "environment": {"tokenizers": __import__("tokenizers").__version__,
                    "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())},
}
(REPORT_DIR / "optimization_sweep.json").write_text(json.dumps(report, ensure_ascii=False, indent=2),
                                                   encoding="utf-8")

# --- Markdown ---------------------------------------------------------------
md = ["# Étape 2 — Balayage d'optimisation (métrique officielle)", "",
      f"*Dataset `{DATASET_NAME}` @ `{DATASET_REVISION}` — train utilisé : "
      f"{sum(len(v) for v in train_by_lang.values()):,} textes, validation : {len(val_rows):,} lignes.*", "",
      "## Classement", "",
      "| Config | Score ↓ | Guardrail | Hausa | Swahili | Yoruba | Amharic | UNK | lossy | vocab | en | fr | budget |",
      "|---|---:|---|---:|---:|---:|---:|---:|---:|---:|---:|---:|---:|"]
for r in sorted(results, key=lambda x: x["score"]):
    md.append(f"| `{r['name']}` | {r['score']:.4f} | {'PASS' if r['guardrail_pass'] else 'FAIL'} | "
              f"{r['penalised']['ha']:.4f} | {r['penalised']['sw']:.4f} | {r['penalised']['yo']:.4f} | "
              f"{r['penalised']['am']:.4f} | {r['unk_total']} | {r['lossy_rows']:,} | {r['vocab_size']} | "
              f"{r['fertility']['en']:.4f} | {r['fertility']['fr']:.4f} | {r['guardrail_budget']:.4f} |")
md += ["", "## Fertility brute par langue", "",
       "| Config | " + " | ".join(LANGUAGES) + " |", "|---|" + "---:|" * len(LANGUAGES)]
for r in sorted(results, key=lambda x: x["score"]):
    md.append(f"| `{r['name']}` | " + " | ".join(f"{r['fertility'][l]:.4f}" for l in LANGUAGES) + " |")
md += ["", "## Décision", "",
       f"- Baseline de référence : **{BASELINE_REFERENCE_SCORE:.4f}**",
       f"- **Meilleure configuration (guardrail OK) : `{best['name']}` — score {best['score']:.4f}** "
       f"({BASELINE_REFERENCE_SCORE - best['score']:+.4f})",
       f"- Guardrail : budget {best['guardrail_budget']:.4f}, en {best['fertility']['en']:.4f}, "
       f"fr {best['fertility']['fr']:.4f} → {'PASS' if best['guardrail_pass'] else 'FAIL'}",
       f"- Modèle : `{best_tokenizer_path}`", "",
       "## Configurations testées", ""]
for r in results:
    md.append(f"- `{r['name']}` — {r['note']} — score {r['score']:.4f} — "
              f"guardrail {'PASS' if r['guardrail_pass'] else 'FAIL'}")
md.append("")
(REPORT_DIR / "optimization_sweep.md").write_text("\n".join(md), encoding="utf-8")
print("Rapports :", REPORT_DIR / "optimization_sweep.json", "|", REPORT_DIR / "optimization_sweep.md")

## 9. Vérification avec le **checker officiel** du challenge

On télécharge `starter/utils.py` du dépôt officiel et on exécute `profile_submission` sur le
tokenizer gagnant, avec le split de validation comme données. C'est **le même code** que celui
utilisé pour valider les soumissions.

In [ ]:
# =============================================================================
# 9. Checker officiel (starter/utils.py du dépôt du challenge)
# =============================================================================
OFFICIAL_UTILS_URL = ("https://raw.githubusercontent.com/aims-ai-research-foundations/"
                      "airf-multilingual-tokenizer-challenge/main/starter/utils.py")
utils_path = OUTPUT_ROOT / "utils.py"
if not utils_path.exists():
    import urllib.request
    try:
        urllib.request.urlretrieve(OFFICIAL_UTILS_URL, utils_path)
        print("utils.py officiel téléchargé")
    except Exception as exc:
        print("Téléchargement impossible :", exc)

official_report = None
if utils_path.exists():
    import importlib, sys
    sys.path.insert(0, str(OUTPUT_ROOT))
    import utils as official_utils
    importlib.reload(official_utils)
    official_report = official_utils.profile_submission(
        candidate_path, data=pd.DataFrame(val_rows, columns=["language", "text"]))
else:
    print("Checker officiel indisponible — utilisation des contrôles intégrés.")
    print(json.dumps(validate_tokenizer_file(candidate_path), indent=2, ensure_ascii=False))

# Contrôles intégrés complémentaires (équivalents à competition/validation.py)
checks = validate_tokenizer_file(candidate_path)
print("\nContrôles officiels intégrés :", checks["checks"])
print("Erreurs :", checks["errors"] or "aucune")
print("Langues non lossless (round-trip) :", checks["lossy_languages"] or "aucune (lossless)")

## 10. Dossier de soumission `submissions/<slug>/`

Génère le dossier attendu par le challenge (voir `CONTRIBUTING.md` du dépôt officiel) :

```text
submissions/maick-dane-nkou/
├── tokenizer.json    obligatoire
├── metadata.yml      obligatoire
├── notebook.ipynb    obligatoire avant la date limite
└── README.md         optionnel (approche)
```

`notebook.ipynb` est joint **automatiquement** (copie du notebook présent sur le disque — Colab
ou dépôt cloné — sinon téléchargement depuis le dépôt). Un dossier sans `notebook.ipynb`
**disqualifie** l'équipe : le contrôle est donc explicite en fin de cellule.

Participation **individuelle** : le règlement autorise les candidatures solo (la contrainte des
deux nationalités ne concerne que les équipes). Le dossier ne peut contenir **que** ces quatre
noms de fichiers — aucun autre fichier, aucun symlink.

In [ ]:
# =============================================================================
# 10. Génération du dossier de soumission
# =============================================================================
import re
import shutil

# --- Métadonnées (participation individuelle) -------------------------------
TEAM_NAME = "Maick Dane Nkou"           # <= 80 caractères, apparaît sur le leaderboard
MEMBERS = ["Maick Dane Nkou"]           # <= 6 membres
AFFILIATION = "AIMS SOUTH AFRICA"       # optionnel (<= 120 caractères)
SLUG = "maick-dane-nkou"                # dossier : minuscules kebab-case (aligné sur TEAM_NAME)
NOTEBOOK_GLOBS = [                      # où chercher notebook.ipynb, dans l'ordre
    "*.ipynb",                                        # Colab : à côté du notebook en cours
    "notebooks/02_optimization_sweep.ipynb",          # dépôt cloné (ce notebook)
]
NOTEBOOK_URL = (                        # dernier recours : téléchargement depuis le dépôt
    "https://raw.githubusercontent.com/maick-code/tokenizer/"
    "arena/01a0889d-tokenizer/notebooks/02_optimization_sweep.ipynb"
)

assert re.fullmatch(r"[a-z0-9]+(?:-[a-z0-9]+)*", SLUG), "slug invalide (minuscules kebab-case)"
assert SLUG != "baseline", "le dossier 'baseline' est réservé"
assert len(MEMBERS) <= 6, "6 membres maximum"
assert TEAM_NAME.strip() and len(TEAM_NAME.strip()) <= 80, "nom d'équipe invalide"

submission_dir = SUBMISSIONS_DIR / SLUG

# Nettoyage : le checker officiel refuse tout fichier inattendu
ALLOWED_FILES = {"tokenizer.json", "metadata.yml", "notebook.ipynb", "README.md"}
if submission_dir.is_dir():
    for item in sorted(submission_dir.iterdir()):
        if item.name not in ALLOWED_FILES:
            print("supprimé (fichier non autorisé) :", item)
            shutil.rmtree(item) if item.is_dir() else item.unlink()
submission_dir.mkdir(parents=True, exist_ok=True)

# --- tokenizer.json ---------------------------------------------------------
shutil.copy2(best_tokenizer_path, submission_dir / "tokenizer.json")

# --- description exacte de la configuration gagnante ------------------------
PRE_NAMES = {"whitespace_split": "WhitespaceSplit", "whitespace": "Whitespace",
             "byte_level": "ByteLevel"}
pre_name = PRE_NAMES.get(best["config"].get("pre"), best["config"].get("pre"))
boost = best["config"].get("boost") or {}
oversample = (f" with {'/'.join(sorted(boost))} oversampled x{max(boost.values())}"
              if boost else "")
byte_clause = (" with byte fallback (256 <0xXX> tokens, no [UNK] possible)"
               if best["config"].get("byte_fallback") else "")
approach = (
    f"BPE {round(best['vocab_size'] / 1000)}k ({best['name']}): NFC, {pre_name} pre-tokenization{byte_clause}, "
    f"trained only on the official train split{oversample}. Validation score {best['score']:.4f} "
    f"(baseline BPE 10k {BASELINE_REFERENCE_SCORE:.4f}); EN/FR guardrail "
    f"{'PASS' if best['guardrail_pass'] else 'FAIL'}."
)[:240]
assert len(approach) <= 240, "approach : 240 caractères maximum"

# --- metadata.yml -----------------------------------------------------------
metadata = {
    "team": TEAM_NAME,
    "members": MEMBERS,
    "affiliation": AFFILIATION[:120],
    "approach": approach,
}
try:
    import yaml
    (submission_dir / "metadata.yml").write_text(
        yaml.safe_dump(metadata, allow_unicode=True, sort_keys=False), encoding="utf-8")
    print("metadata.yml écrit (YAML).")
except ImportError:
    lines = [f"team: {TEAM_NAME}", "members:"]
    lines += [f"  - {member}" for member in MEMBERS]
    lines += [f"affiliation: {AFFILIATION}", f"approach: {approach}"]
    (submission_dir / "metadata.yml").write_text("\n".join(lines) + "\n", encoding="utf-8")
    print("metadata.yml écrit (YAML manuel).")

# --- README.md (approche) ---------------------------------------------------
size_of_byte_vocab = ("yes — the 256 `<0xXX>` tokens cover every possible UTF-8 character, so\n"
                     "  no `[UNK]` can ever be emitted" if best["config"].get("byte_fallback")
                     else "no — the few `[UNK]` left are rare non-African residues of the source\n"
                          "  text (Arabic presentation forms, CJK, kana, hangul, emoji)")
result_rows = "\n".join(
    f"| {LANGUAGE_NAMES[l]} | {best['fertility'][l]:.4f} | {best['unk_rate'][l]:.6f} | "
    f"{best['penalised'][l]:.4f} |" for l in LANGUAGES)
(submission_dir / "README.md").write_text(
    f"# {TEAM_NAME}\n\n"
    f"Individual entry — tokenizer `{best['name']}`.\n\n"
    f"## Approach\n\n"
    f"- Model: BPE with `[UNK]` as unknown token; vocabulary {best['vocab_size']:,} / 10,000\n"
    f"- Normalizer: NFC (no ASCII folding, no accent stripping, no lowercasing)\n"
    f"- Pre-tokenizer: `{pre_name}` — punctuation stays attached to its word, so no token is\n"
    f"  spent on isolated `,` `.` `)` …\n"
    f"- Byte fallback: {size_of_byte_vocab}\n"
    f"- Training corpus: official `train` split only, balanced round-robin over the six\n"
    f"  languages{oversample}; no external corpus and no pre-trained tokenizer\n"
    f"- Post-processor / decoder: none\n"
    f"- Built with `tokenizers==0.22.1`\n\n"
    f"## Results (official validation split, official metric)\n\n"
    f"| Language | Fertility | UNK rate | Score |\n|---|---:|---:|---:|\n{result_rows}\n\n"
    f"- **Score (mean of ha, sw, yo, am): {best['score']:.4f}** — baseline BPE 10k "
    f"{BASELINE_REFERENCE_SCORE:.4f}, i.e. a gain of "
    f"{BASELINE_REFERENCE_SCORE - best['score']:+.4f} "
    f"({100 * (BASELINE_REFERENCE_SCORE - best['score']) / BASELINE_REFERENCE_SCORE:.1f} %)\n"
    f"- Context guardrail EN/FR: **{'PASS' if best['guardrail_pass'] else 'FAIL'}** "
    f"(budget {best['guardrail_budget']:.4f}, en {best['fertility']['en']:.4f}, "
    f"fr {best['fertility']['fr']:.4f})\n"
    f"- UNK emitted on validation: {best['unk_total']}\n\n"
    f"## Files\n\n"
    f"- `tokenizer.json` — the submitted tokenizer\n"
    f"- `metadata.yml` — team metadata\n"
    f"- `notebook.ipynb` — the notebook that built this tokenizer "
    f"(`notebooks/02_optimization_sweep.ipynb`)\n"
    f"- `README.md` — this file\n",
    encoding="utf-8")

# --- notebook.ipynb (obligatoire avant la date limite) ----------------------
def attach_notebook(target_dir):
    """Copie le notebook depuis le disque (Colab puis dépôt cloné), sinon le télécharge."""
    for pattern in NOTEBOOK_GLOBS:
        matches = sorted(OUTPUT_ROOT.glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)
        if matches:
            shutil.copy2(matches[0], target_dir / "notebook.ipynb")
            print(f"notebook.ipynb : copié depuis {matches[0]}")
            return True
    try:
        import urllib.request
        urllib.request.urlretrieve(NOTEBOOK_URL, target_dir / "notebook.ipynb")
        print("notebook.ipynb : téléchargé depuis le dépôt")
        return True
    except Exception as exc:
        print("notebook.ipynb MANQUANT :", exc)
        print("  -> un dossier sans notebook.ipynb est DISQUALIFIÉ (règlement du challenge)")
        print("  -> déposez le notebook dans", target_dir)
        return False


notebook_attached = attach_notebook(submission_dir)

print("\nDossier de soumission :", submission_dir)
for path in sorted(submission_dir.iterdir()):
    print(f"  {path.name} ({path.stat().st_size:,} octets)")
print("\nFichiers autorisés :", sorted(ALLOWED_FILES))
print("metadata.yml :", metadata)
print("Prêt pour la PR officielle :", "OUI" if notebook_attached else "presque (notebook.ipynb à ajouter)")


## 11. Prochaines étapes

1. **Remplir** `TEAM_NAME`, `MEMBERS`, `SLUG` (cellule 10) puis relancer la cellule 10.
2. **`notebook.ipynb`** : joint automatiquement par la cellule 10 (c'est ce notebook-ci).
   Vérifier la ligne finale « Prêt pour la PR officielle : OUI ».
3. **Publier sur GitHub** :
   - *fork* de `aims-ai-research-foundations/airf-multilingual-tokenizer-challenge` ;
   - branche nommée exactement **`submission`** ;
   - `submissions/<slug>/` ajouté, puis *Pull Request* vers le dépôt officiel.
4. **PR** : cocher la checklist du template et décrire l'approche.

Rappels de contrainte (vérifiés dans le code officiel) :
- une PR ne doit modifier **que** `submissions/<slug>/` (3 niveaux de chemin, un seul slug) ;
- le dossier ne peut contenir que `tokenizer.json`, `metadata.yml`, `notebook.ipynb`, `README.md` ;
- pas de symlink ; `metadata.yml` ≤ 16 KiB ; `tokenizer.json` ≤ 20 MiB ; vocab ≤ 10 000 ;
- **`tokenizers==0.22.1`** (le workflow du challenge installe cette version via `uv.lock`) ;
- évaluation : temps ≤ 5× celui de la baseline (les tie-breaks se font sur la vitesse).

In [ ]:
# =============================================================================
# 11. Récapitulatif final
# =============================================================================
print("Balayage d'optimisation terminé.\n")
print(f"Baseline de référence : {BASELINE_REFERENCE_SCORE:.4f}")
print(f"Meilleure configuration éligible : {best['name']} -> {best['score']:.4f} "
      f"({BASELINE_REFERENCE_SCORE - best['score']:+.4f})")
print(f"Guardrail : {'PASS' if best['guardrail_pass'] else 'FAIL'}")
print()
print("Artefacts :")
print(f"  - {best_tokenizer_path}")
print(f"  - {REPORT_DIR / 'optimization_sweep.json'}")
print(f"  - {REPORT_DIR / 'optimization_sweep.md'}")
print(f"  - {submission_dir} (soumission)")
if official_report is not None:
    print("\nChecker officiel :", "READY FOR SUBMISSION" if official_report.get("valid")
          else f"NOT READY -> {official_report.get('errors')}")

## 12. Publier les artefacts sur GitHub (script + token)

Le balayage a produit `models/optimized_*/`, `reports/optimization_sweep.{json,md}` et le dossier
`submissions/<slug>/`. La cellule suivante **écrit le script de publication** (contenu identique à
`scripts/push_artifacts_to_github.py` du dépôt) et la dernière l'**exécute dans le processus du
notebook** : un **champ masqué** s'affiche pour coller le token GitHub (portée `repo`).

`--include-submissions` publie aussi le dossier de soumission. Le script refuse de publier des
artefacts non conformes (run sur données synthétiques) et masque le token dans toutes les sorties.

### Cellule « script de publication »

La cellule suivante **écrit le script** `push_artifacts_to_github.py` dans le répertoire courant
(contenu identique à `scripts/push_artifacts_to_github.py` du dépôt), et celle d'après **l'exécute
dans le processus du notebook** — indispensable pour que le **champ masqué Colab** fonctionne et
pour que le script voie les Secrets Colab.

Le token est demandé par saisie masquée (ou lu dans le secret Colab `GITHUB_TOKEN` s'il existe).
Il n'est jamais affiché, jamais écrit sur disque, jamais commité.

Ce que le script publie : `models/**`, `reports/**` (+ `submissions/**` avec `--include-submissions`),
sur la branche `arena/01a0889d-tokenizer` (`main` reste intacte).

In [33]:
%%writefile push_artifacts_to_github.py
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""Publie les artefacts du challenge vers votre dépôt GitHub (méthode 2 : token).

Le token est fourni par saisie **masquée** (recommandé), par variable
d'environnement, ou par le secret Colab ``GITHUB_TOKEN``. Il n'est **jamais**
affiché, jamais écrit sur disque, jamais commité : toutes les sorties passent par
``redact()``.

Ce qui est publié (par défaut) :
    models/**      tokenizer(s) entraîné(s)
    reports/**     rapports JSON / Markdown
    submissions/** (avec --include-submissions) dossier de soumission

Exemples
--------
Colab — recommandé (dans une cellule Python, champ masqué actif) :
    import sys, runpy
    sys.argv = ["push_artifacts_to_github.py", "--source", "/content", "--include-submissions"]
    try:
        runpy.run_path("/content/push_artifacts_to_github.py", run_name="__main__")
    except SystemExit as exc:
        print("code de sortie :", exc.code)

Colab — avec !python : un sous-processus n'a ni champ masqué ni Secrets, il faut
fournir le token autrement (secret exporté dans l'environnement, ou --token-file) :
    !python scripts/push_artifacts_to_github.py --source /content

Colab, en incluant le dossier de soumission :
    !python scripts/push_artifacts_to_github.py --source /content --include-submissions

Local :
    python scripts/push_artifacts_to_github.py --repo . --source .

Vérifier sans rien publier :
    python scripts/push_artifacts_to_github.py --source . --no-push

Créer explicitement une branche inexistante :
    python scripts/push_artifacts_to_github.py --source . --branch nouvelle-branche --create-branch

Publier sur une autre branche / un autre dépôt :
    python scripts/push_artifacts_to_github.py --source . --branch main \
        --repo-url https://github.com/<user>/<repo>.git
"""

from __future__ import annotations

import argparse
import getpass
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

DEFAULT_REPO_URL = "https://github.com/maick-code/tokenizer.git"
DEFAULT_BRANCH = "arena/01a0889d-tokenizer"   # branche de travail (main reste intacte)
DEFAULT_MESSAGE = "Artifacts: tokenizer.json + reports (run Colab)"
ARTIFACT_DIRS = ("models", "reports")
EXCLUDE_DIR_NAMES = {"__pycache__", ".ipynb_checkpoints", ".git"}
EXCLUDE_SUFFIXES = (".pyc", ".pyo", ".zip", ".tmp", ".log")

EXIT_OK, EXIT_ERROR, EXIT_MISSING, EXIT_UNSAFE = 0, 1, 2, 3


# --------------------------------------------------------------------------- #
# Utilitaires
# --------------------------------------------------------------------------- #
def log(message: str = "") -> None:
    print(message, flush=True)


def die(message: str, code: int) -> "NoReturn":  # noqa: F821
    log(f"\nERREUR : {message}")
    raise SystemExit(code)


def redact(text: str, token: str | None) -> str:
    """Supprime toute trace du token d'une sortie."""
    if not text:
        return ""
    if token:
        text = text.replace(token, "***")
    return text


def clone_dir_default() -> Path:
    if os.path.isdir("/content"):          # Google Colab
        return Path("/content/tokenizer")
    return Path.cwd() / ".push_clone"


# --------------------------------------------------------------------------- #
# Token
# --------------------------------------------------------------------------- #
def token_from_colab_secret() -> str | None:
    try:
        from google.colab import userdata  # type: ignore

        value = userdata.get("GITHUB_TOKEN")
        return value.strip() if value else None
    except Exception:
        return None


_COLAB_MASKED_FIELD_JS = r"""
new Promise((resolve) => {
  const box = document.createElement('div');
  box.style.cssText = 'font-family:monospace;padding:10px;margin-top:6px;'
                    + 'border:1px solid #c8c8c8;border-radius:6px;display:inline-block';
  const label = document.createElement('span');
  label.textContent = 'Colle ton token GitHub puis valide : ';
  const input = document.createElement('input');
  input.type = 'password';
  input.style.cssText = 'font-size:14px;padding:3px 5px;width:330px';
  const button = document.createElement('button');
  button.textContent = 'Enregistrer';
  button.style.cssText = 'margin-left:8px;padding:3px 12px';
  const done = () => {
    input.disabled = true; button.disabled = true;
    const value = input.value; box.remove(); resolve(value);
  };
  button.addEventListener('click', done);
  input.addEventListener('keydown', (event) => { if (event.key === 'Enter') done(); });
  box.appendChild(label); box.appendChild(input); box.appendChild(button);
  document.body.appendChild(box);
  input.focus();
})
"""


def token_from_colab_masked_field() -> str | None:
    """Champ de saisie masqué natif Colab (nécessite d'exécuter le script EN PROCESSUS).

    Fonctionne quand le script est lancé dans une cellule Python (``runpy``), pas
    avec ``!python`` : un sous-processus n'a pas accès à l'interface du notebook.
    """
    try:
        from google.colab import output  # type: ignore
    except Exception:
        return None
    try:
        value = output.eval_js(_COLAB_MASKED_FIELD_JS)
    except Exception as exc:
        log(f"Champ masqué Colab indisponible ({type(exc).__name__}) : repli sur la saisie classique.")
        return None
    if isinstance(value, str) and value.strip():
        log("Token saisi dans le champ masqué Colab (non affiché).")
        return value.strip().strip('"').strip("'")
    return None


def read_token(args: argparse.Namespace) -> str | None:
    """Token par ordre de priorité : --token-file, env, secret Colab, champ masqué Colab, saisie."""
    if args.token_file:
        path = Path(args.token_file)
        if not path.is_file():
            die(f"fichier de token introuvable : {path}", EXIT_ERROR)
        token = path.read_text(encoding="utf-8").strip()
        if token:
            log("Token lu depuis le fichier indiqué (--token-file).")
            return token

    for var in ("GITHUB_TOKEN", "GH_TOKEN"):
        token = os.environ.get(var)
        if token:
            log(f"Token récupéré depuis la variable d'environnement {var}.")
            return token.strip()

    token = token_from_colab_secret()
    if token:
        log("Token récupéré depuis le secret Colab 'GITHUB_TOKEN'.")
        return token

    if args.no_input:
        return None

    token = token_from_colab_masked_field()
    if token:
        return token

    prompt = "Colle ton token GitHub puis Entrée : "
    try:
        token = getpass.getpass(prompt)          # saisie masquée
    except Exception:
        try:
            token = input(prompt)                # repli si getpass indisponible
        except Exception:
            return None
    token = (token or "").strip().strip('"').strip("'")
    if token:
        log(f"Token saisi ({len(token)} caractères, non affiché).")
    return token or None


def authed_url(url: str, token: str | None) -> str:
    """URL https porteuse du token, uniquement pour github.com."""
    if token and url.startswith("https://github.com/"):
        return url.replace("https://", f"https://x-access-token:{token}@")
    return url


# --------------------------------------------------------------------------- #
# Git
# --------------------------------------------------------------------------- #
def git(repo: Path | str | None, *args: str) -> subprocess.CompletedProcess:
    command = ["git"]
    if repo is not None:
        command += ["-C", str(repo)]
    return subprocess.run(command + list(args), capture_output=True, text=True)


def git_or_die(repo: Path | str | None, token: str | None, *args: str,
               what: str = "commande git") -> subprocess.CompletedProcess:
    result = git(repo, *args)
    if result.returncode != 0:
        die(f"{what} a échoué :\n{redact(result.stderr or result.stdout, token).strip()}",
            EXIT_ERROR)
    return result


# --------------------------------------------------------------------------- #
# Artefacts
# --------------------------------------------------------------------------- #
def collect_artifacts(source: Path, include_submissions: bool) -> list[str]:
    """Chemins relatifs (posix) des fichiers à publier, triés."""
    roots = list(ARTIFACT_DIRS) + (["submissions"] if include_submissions else [])
    files: list[str] = []
    for root in roots:
        base = source / root
        if not base.is_dir():
            continue
        for path in sorted(base.rglob("*")):
            if not path.is_file():
                continue
            parts = set(path.relative_to(source).parts)
            if parts & EXCLUDE_DIR_NAMES or path.name.startswith("."):
                continue
            if path.suffix.lower() in EXCLUDE_SUFFIXES:
                continue
            files.append(path.relative_to(source).as_posix())
    return files


def safety_checks(source: Path, files: list[str], force: bool) -> list[str]:
    """Contrôles avant publication. Retourne la liste des avertissements bloquants."""
    import json

    problems: list[str] = []

    baseline = source / "reports" / "baseline_bpe_10k.json"
    if baseline.is_file():
        try:
            status = json.loads(baseline.read_text(encoding="utf-8")).get("status")
            if status != "computed_on_official_dataset":
                problems.append(
                    f"reports/baseline_bpe_10k.json : status = {status!r} "
                    "(run non conforme au dataset officiel)")
        except Exception as exc:
            problems.append(f"reports/baseline_bpe_10k.json illisible : {exc}")

    sweep = source / "reports" / "optimization_sweep.json"
    if sweep.is_file():
        try:
            payload = json.loads(sweep.read_text(encoding="utf-8"))
            rows = (payload.get("dataset") or {}).get("validation_rows")
            if rows != 24_000:
                problems.append(
                    f"reports/optimization_sweep.json : validation_rows = {rows} "
                    "(attendu 24 000 : le balayage n'a pas tourné sur le vrai dataset)")
        except Exception as exc:
            problems.append(f"reports/optimization_sweep.json illisible : {exc}")

    if not any(f.startswith("models/") and f.endswith("tokenizer.json") for f in files):
        problems.append("aucun models/**/tokenizer.json trouvé dans les artefacts")

    if problems and not force:
        log("\n" + "!" * 74)
        log("PUBLICATION REFUSÉE — les artefacts semblent ne pas venir d'un run réel :")
        for problem in problems:
            log(f"  - {problem}")
        log("Corrigez le run, ou relancez avec --force pour publier quand même.")
        log("!" * 74)
        raise SystemExit(EXIT_UNSAFE)

    if problems:
        log("\nAVERTISSEMENT (--force) :")
        for problem in problems:
            log(f"  - {problem}")
    return problems


# --------------------------------------------------------------------------- #
# Programme principal
# --------------------------------------------------------------------------- #
def parse_args(argv: list[str] | None = None) -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Publie models/ et reports/ vers votre dépôt GitHub (méthode token).",
        formatter_class=argparse.RawDescriptionHelpFormatter,
        epilog=__doc__,
    )
    parser.add_argument("--source", default="/content" if os.path.isdir("/content") else ".",
                        help="répertoire contenant models/ et reports/ (défaut : /content ou .)")
    parser.add_argument("--repo", default=None,
                        help="clone git existant du dépôt (sinon clonage automatique)")
    parser.add_argument("--repo-url", default=DEFAULT_REPO_URL, help="URL https du dépôt")
    parser.add_argument("--branch", default=DEFAULT_BRANCH, help="branche cible")
    parser.add_argument("--message", default=DEFAULT_MESSAGE, help="message de commit")
    parser.add_argument("--token-file", default=None,
                        help="lire le token depuis un fichier (évite la saisie)")
    parser.add_argument("--no-input", action="store_true",
                        help="ne jamais demander le token de façon interactive")
    parser.add_argument("--no-push", action="store_true",
                        help="copier et commiter sans pousser")
    parser.add_argument("--include-submissions", action="store_true",
                        help="publier aussi submissions/**")
    parser.add_argument("--zip", action="store_true",
                        help="créer en plus une archive de secours dans --source")
    parser.add_argument("--force", action="store_true",
                        help="publier malgré les avertissements de conformité")
    parser.add_argument("--create-branch", action="store_true",
                        help="autoriser la création de la branche si elle n'existe pas")
    return parser.parse_args(argv)


def main(argv: list[str] | None = None) -> int:
    args = parse_args(argv)
    source = Path(args.source).resolve()
    branch = args.branch
    repo_url = args.repo_url

    log("=" * 74)
    log("Publication des artefacts vers GitHub")
    log("=" * 74)
    log(f"Source      : {source}")
    log(f"Dépôt       : {repo_url}")
    log(f"Branche     : {branch}")
    log(f"Artefacts   : {', '.join(ARTIFACT_DIRS + (('submissions',) if args.include_submissions else ()))}")
    log()

    files = collect_artifacts(source, args.include_submissions)
    if not files:
        die(f"aucun artefact trouvé dans {source} (attendu : models/, reports/)", EXIT_MISSING)

    log(f"{len(files)} fichier(s) à publier :")
    total = 0
    for rel in files:
        size = (source / rel).stat().st_size
        total += size
        log(f"  {size:>12,} o  {rel}")
    log(f"  {'-' * 12}")
    log(f"  {total:>12,} o  total")

    safety_checks(source, files, args.force)

    if args.zip:
        archive = source / "artifacts_backup.zip"
        with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as handle:
            for rel in files:
                handle.write(source / rel, rel)
        log(f"\nArchive de secours : {archive} ({archive.stat().st_size:,} o)")

    token = read_token(args)

    repo = Path(args.repo).resolve() if args.repo else clone_dir_default()

    if not (repo / ".git").exists():
        if not token:
            die("aucun token fourni et pas de clone local : impossible de cloner.", EXIT_ERROR)
        log(f"\nClone de {repo_url} (branche {branch}) dans {repo} ...")
        clone = subprocess.run(
            ["git", "clone", "--branch", branch, authed_url(repo_url, token), str(repo)],
            capture_output=True, text=True)
        if clone.returncode != 0:
            die("clonage impossible (token invalide, branche inexistante ou réseau) :\n"
                f"{redact(clone.stderr or clone.stdout, token).strip()}", EXIT_ERROR)
        log("clone : OK")
    else:
        log(f"\nClone existant réutilisé : {repo}")

    if token:
        check = git(repo, "ls-remote", "--heads", authed_url(repo_url, token), branch)
        if check.returncode != 0:
            die("authentification refusée : vérifiez la portée `repo` du token,"
                " sa date d'expiration et le nom de la branche.", EXIT_ERROR)
        log("authentification : OK")

    # La branche cible doit exister : sans ce contrôle, une faute de frappe
    # créerait silencieusement une nouvelle branche distante.
    exists = git(None, "ls-remote", "--heads",
                 authed_url(repo_url, token) if token else repo_url, branch)
    if exists.returncode == 0 and not exists.stdout.strip():
        if args.create_branch:
            log(f"branche '{branch}' absente du dépôt : elle sera créée (--create-branch).")
        else:
            die(f"la branche '{branch}' n'existe pas sur {repo_url}.\n"
                "Vérifiez le nom (--branch), ou utilisez --create-branch pour la créer.",
                EXIT_ERROR)
    elif exists.returncode != 0 and not token:
        log("(impossible de vérifier la branche sans token : le push tranchera.)")

    for rel in files:
        destination = repo / rel
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source / rel, destination)
    log(f"{len(files)} fichier(s) copié(s) dans le clone.")

    git(repo, "config", "user.name", "Artifact Publisher")
    git(repo, "config", "user.email", "publisher@users.noreply.github.com")
    for root in {Path(rel).parts[0] for rel in files}:
        git_or_die(repo, token, "add", root, what=f"git add {root}")

    commit = git(repo, "commit", "-m", args.message)
    if commit.returncode == 0:
        log("commit : OK")
    elif "nothing to commit" in (commit.stdout + commit.stderr):
        log("commit : rien de nouveau (artefacts identiques)")
    else:
        die(f"commit impossible :\n{redact(commit.stderr or commit.stdout, token).strip()}",
            EXIT_ERROR)

    if args.no_push:
        log("\n--no-push : publication non effectuée. Commande manuelle :")
        log(f"  git -C {repo} push {repo_url} HEAD:{branch}")
        return EXIT_OK

    if not token:
        log("\nAucun token : publication non effectuée. Commande manuelle :")
        log(f"  git -C {repo} push {repo_url} HEAD:{branch}")
        return EXIT_OK

    push = git(repo, "push", authed_url(repo_url, token), f"HEAD:{branch}")
    if push.returncode != 0:
        die(f"push refusé :\n{redact(push.stderr or push.stdout, token).strip()}", EXIT_ERROR)

    log("push : OK")
    log()
    log(f"Publié sur {repo_url} (branche {branch}).")
    if "github.com" in repo_url:
        slug = repo_url.rstrip("/").removesuffix(".git")
        log(f"Vérifiez : {slug}/tree/{branch}")
    return EXIT_OK


if __name__ == "__main__":
    raise SystemExit(main())


Overwriting push_artifacts_to_github.py


In [ ]:
# =============================================================================
# Exécution du script de publication (champ masqué Colab actif)
# =============================================================================
import runpy
import sys
from pathlib import Path

SCRIPT_PATH = Path("push_artifacts_to_github.py").resolve()
assert SCRIPT_PATH.is_file(), "la cellule %%writefile ci-dessus doit être exécutée d'abord"

# --source : racine des artefacts (contient models/ et reports/)
sys.argv = [
    "push_artifacts_to_github.py",
    "--source", str(OUTPUT_ROOT),
    "--branch", "arena/01a0889d-tokenizer",
    "--include-submissions",
]

print("Exécution :", SCRIPT_PATH)
print("Arguments :", " ".join(sys.argv[1:]))
print("Un champ masqué « Colle ton token GitHub puis valide » va s'afficher.")
print()
try:
    runpy.run_path(str(SCRIPT_PATH), run_name="__main__")
except SystemExit as exc:
    print()
    print("Code de sortie du script :", exc.code,
          "| 0 = OK, 1 = erreur, 2 = artefacts manquants, 3 = publication refusée")